In [0]:
import dlt
import pyspark.sql.functions as F

catalog = "meridian_tnn" 

@dlt.view(name="bronze_returns_stream")
def bronze_returns_stream():
    return dlt.read_stream("bronze.bronze_store_returns") # Add the schema prefix here too

@dlt.view(name="sales_lookup")
def sales_lookup():
    return dlt.read("bronze.bronze_store_sales")

# 2. Cleansing, Casting, and Orphan Detection
@dlt.view(name="returns_cleansed")
def returns_cleansed():
    df_returns = dlt.read_stream("bronze_returns_stream")
    df_sales = dlt.read("sales_lookup")
    
    joined = df_returns.join(
        F.broadcast(df_sales),
        (df_returns.sr_item_sk == df_sales.ss_item_sk) & 
        (df_returns.sr_ticket_number == df_sales.ss_ticket_number),
        "left"
    )
    
    return joined.select(
        df_returns.sr_item_sk.cast("int").alias("item_sk"),
        df_returns.sr_customer_sk.cast("int").alias("customer_sk"),
        df_returns.sr_ticket_number.cast("int").alias("ticket_number"),
        df_returns.sr_return_quantity.cast("int").alias("return_quantity"),
        df_returns.sr_refunded_cash.cast("decimal(10,2)").alias("refunded_cash"),
        df_returns.sr_reason_sk.cast("int").alias("reason_sk"),
        
        # EXTRACT BATCH NUMBER USING source_file_path FROM BRONZE
        F.regexp_extract(df_returns.source_file_path, r"batch_([0-9]+)", 1).cast("int").alias("batch_number"),
        
        F.when(df_sales.ss_item_sk.isNull(), True).otherwise(False).alias("is_orphan_return"),
        
        F.when(
            F.col("sr_item_sk").isNull() | 
            (F.col("sr_return_quantity") <= 0) | 
            (F.col("sr_refunded_cash") < 0), 
            False
        ).otherwise(True).alias("is_valid")
    )

# 3. Valid Silver Table (Deduplicated)
@dlt.table(
    name="silver.silver_store_returns",
    comment="Validated, deduplicated fact table"
)
@dlt.expect("valid_quantity", "return_quantity > 0")       
@dlt.expect("valid_item", "item_sk IS NOT NULL")          
@dlt.expect("valid_refund", "refunded_cash >= 0")          
def silver_store_returns():
    return (
        dlt.read_stream("returns_cleansed")
        .filter("is_valid = True")
        .drop("is_valid")
        .dropDuplicates(["ticket_number", "item_sk"]) 
    )

# 4. Quarantine Table (Ops schema)
@dlt.table(
    name="ops.quarantine_store_returns",
    comment="Invalid records dropped from Silver",
    table_properties={"pipelines.autoOptimize.targetSchema": f"{catalog}.ops"}
)
def quarantine_store_returns():
    return (
        dlt.read_stream("returns_cleansed")
        .filter("is_valid = False")
        .drop("is_valid")
    )

# 5. Historise Dimension: Read Bronze Reason as Stream
@dlt.view(name="bronze_reason_stream")
def bronze_reason_stream():
    return dlt.read_stream("bronze.bronze_reason")

# 2. Create the streaming table with the silver. prefix
dlt.create_streaming_table("silver.silver_reason_history")

# 3. Apply changes with the EXACT same target name
dlt.apply_changes(
    target="silver.silver_reason_history",     # MUST MATCH THE NAME ABOVE
    source="bronze_reason_stream", 
    keys=["r_reason_sk"],
    sequence_by=F.col("batch_number"), 
    apply_as_deletes=None,
    except_column_list=["source_file_path", "ingestion_time"],
    track_history_column_list=["r_reason_id", "r_reason_desc"],
    stored_as_scd_type=2
)

# 6. Silver Store Dimension (Needed for Row-Filter Security)
@dlt.table(
    name="silver.silver_store",
    comment="Clean store dimension for row-level security"
)
def silver_store():
    # Use read_stream if you used Autoloader in Bronze, otherwise use read
    return dlt.read_stream("bronze.bronze_store").dropDuplicates(["s_store_sk"])

# 7. Silver Item Dimension (Needed for Visual Data Prep)
@dlt.table(
    name="silver.silver_item",
    comment="Clean item dimension for category grouping"
)
def silver_item():
    return dlt.read_stream("bronze.bronze_item").dropDuplicates(["i_item_sk"])

# 8. Silver Customer Dimension (Needed for Masking)
@dlt.table(
    name="silver.silver_customer",
    comment="Clean customer dimension for PII masking"
)
def silver_customer():
    return dlt.read_stream("bronze.bronze_customer").dropDuplicates(["c_customer_sk"])